In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from dotenv import load_dotenv

In [14]:
import os
load_dotenv()
EXCHANGE_RATE_API = os.getenv("EXCHANGE_RATE_API")


In [19]:
# tool create
# 2 tools -> 1 for conversion ; 2 multiplication with value

@tool
def get_conversion_factor(base_currency:str)->float :
    '''This function fetches the currency conversion factor b/w base currency and target curency '''
    url = f"https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_API}/latest/{base_currency}"
    response = requests.get(url)
    return response.json()
@tool
def convert(base_currency_value:int , conversion_rate:float)-> float:
    '''Given a currency conversion rate this function calculates the target currency value from a given base currency value'''

    return base_currency_value * conversion_rate


In [18]:
result = get_conversion_factor.invoke({'base_currency':'USD'})
result

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1784937602,
 'time_last_update_utc': 'Sat, 25 Jul 2026 00:00:02 +0000',
 'time_next_update_unix': 1785024002,
 'time_next_update_utc': 'Sun, 26 Jul 2026 00:00:02 +0000',
 'base_code': 'USD',
 'conversion_rates': {'USD': 1,
  'AED': 3.6725,
  'AFN': 65.9437,
  'ALL': 82.1789,
  'AMD': 366.1235,
  'ANG': 1.79,
  'AOA': 925.523,
  'ARS': 1494.3744,
  'AUD': 1.4319,
  'AWG': 1.79,
  'AZN': 1.7005,
  'BAM': 1.7191,
  'BBD': 2.0,
  'BDT': 123.4131,
  'BGN': 1.7191,
  'BHD': 0.376,
  'BIF': 2994.3398,
  'BMD': 1.0,
  'BND': 1.2909,
  'BOB': 10.9917,
  'BRL': 5.0781,
  'BSD': 1.0,
  'BTN': 96.6133,
  'BWP': 14.1631,
  'BYN': 2.8739,
  'BZD': 2.0,
  'CAD': 1.4088,
  'CDF': 2269.5068,
  'CHF': 0.8179,
  'CLF': 0.02387,
  'CLP': 943.6232,
  'CNH': 6.7721,
  'CNY': 6.7819,
  'COP': 3213.0442,
  'CRC': 454.114,
  'CUP': 24.0,
  'CVE': 

In [20]:
convert.invoke({'base_currency_value':10,'conversion_rate':96.6138})

966.1379999999999

In [21]:
#Step 2 -> tool binding

In [22]:
llm = ChatGoogleGenerativeAI(model = 'gemini-3.6-flash')

In [26]:
llm_with_tools= llm.bind_tools([convert,get_conversion_factor])

In [27]:
# tool calling
messages=[HumanMessage('What is the conversion factor between usd and inr and based on that can u convert 10 usd to inr')]

In [29]:
ai_msg = llm_with_tools.invoke(messages)
ai_msg

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD"}'}, '__gemini_function_call_thought_signatures__': {'EDdJZDYU': 'EsUHCsIHARFNMg94xgXxwpryszVtLKl5HAH9y1kXV6a9lrJd79aAlUm7i/QCm78CSHXf9EJy0sB4qjTEZDJqF1TVuqlDDEcLVYRzoS+vKFeF563sxX1vepcKZrUZCBB12dQaCiE7P6F3K82lDJpOdknct672akeMgTqhRYD9+lRrEALEz4H1UybTMZ8MlUwx+TPxNtjv0TaOH2t+ybtfKslmX2VKwnFtwTBVaNFTkDTrgB1X6qkY5mUwo/kRDsshHtuKLfB5+4AzNVrAEFBcUcdT5WQqChMGKWkYG2uKHgI1CRu9e9VzqpWY/ttUzfp18CVAhFn2Ejdrv8Jv11gw+ME87SyGJ6IzIbKcwm5DGbzDW8jatAugyBQtYZlixLVSc4nH+x4O38gF4cb+xt8HCd8+obcYQchNjqyuaKm7aXS3HVYQn45jYB9wGkGibozcCVIs+/2ojl2GyhHi+4CFYoqUkOsmhOlXT43Dr/R7ZXB5gYsk+Gzj96Ge0NJWbCRp+4fcr0T3NxT0lDAeEGv4HtmVRYkwKePKctEFnGxh4KdkRZDdDifZjcpf5EigMLvwwTVouKa1d6P6DvFNpV6su/1/wNF/g0EsLSJ7cHES3+vy0j6T/R8C1rWoGD9MwXVHzGRCRyhqLjBmFcF2wXqrdzIhYXYKKgZt4WNveUMrqqF9XxJ/qoK91BDhvvM96mzzvq5pJOnZAB/yrqxt9VIhMBAw1CGmgr/zRvEunTV88/dCv9ZwuzOQro0WK7t4NBQeu3AXc4EhnEA2g4uxsH52j4sGVYmygjpLKRo/s